# SGGF-Net Training on HIT-UAV Dataset

This notebook trains SGGF-Net for UAV image object detection using the HIT-UAV infrared thermal dataset.


## Step 1: Mount Google Drive (for checkpoints) and Clone Repository


# Mount Google Drive (only for checkpoints, dataset comes from GitHub)
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository (includes dataset in data/hit-uav/)
!git clone https://github.com/HarishSankarK/SGGF-Net.git
%cd SGGF-Net

# Verify dataset is included
!ls -la data/hit-uav/ 2>/dev/null && echo "✓ Dataset found in repository!" || echo "⚠ Dataset not found"


In [ ]:
# Verify we're in the right directory
import os
print(f"Current directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")


## Step 2: Install Dependencies


In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install -r requirements.txt


## Step 3: Verify Dataset

The dataset is already included in the repository at `data/hit-uav/`. No need to upload!


In [ ]:
# Dataset is already in the repository!
# Just verify it's there
import os

if os.path.exists('data/hit-uav'):
    print("✓ Dataset found in data/hit-uav/")
    print(f"  Images: {len([f for f in os.listdir('data/hit-uav/images/train') if f.endswith('.jpg')])} training images")
    print(f"  Labels: {len([f for f in os.listdir('data/hit-uav/labels/train') if f.endswith('.txt')])} training labels")
    print("\nDataset structure:")
    !ls -la data/hit-uav/
else:
    print("⚠ Dataset not found. Make sure you've pushed it to GitHub.")


### Alternative: If Dataset Not in GitHub (Fallback)


In [ ]:
# Only use this if dataset is NOT in the GitHub repository
# Uncomment and use if needed:

# from google.colab import files
# import zipfile
# 
# # Create data directory
# !mkdir -p data
# 
# # Option 1: Upload from Drive
# # !cp -r /content/drive/MyDrive/hit-uav ./data/hit-uav
# 
# # Option 2: Upload zip file
# # uploaded = files.upload()
# # for filename in uploaded.keys():
# #     if filename.endswith('.zip'):
# #         with zipfile.ZipFile(filename, 'r') as zip_ref:
# #             zip_ref.extractall('data/')
# #         print(f'✓ Extracted {filename} to data/')

print("Dataset should be in the repository. If not, uncomment the code above.")


## Step 4: Verify Setup (GPU and Dataset)


In [ ]:
# Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')

print("\n" + "="*50)
print("Testing dataset loading...")
print("="*50)

# Test dataset loading (using data/ folder)
!python scripts/test_hituav_dataset.py


## Step 5: Start Training


In [ ]:
# Setup checkpoint directory in Google Drive
import os
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(drive_checkpoint_dir, exist_ok=True)
print(f'Checkpoints will be saved to: {drive_checkpoint_dir}')

# Train the model (checkpoints saved to Drive)
# Optimized for speed: AMP enabled by default, reduced validation frequency
# If you get OOM errors, reduce batch_size to 1 or max_size to 800
!python scripts/train.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --num_classes 6 \
    --batch_size 2 \
    --num_epochs 50 \
    --lr 0.005 \
    --max_size 1024 \
    --checkpoint_dir {drive_checkpoint_dir} \
    --device cuda \
    --val_freq 10 \
    --compile


## Step 6: Evaluate Model


In [ ]:
# Evaluate on test set (using checkpoint from Drive)
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

!python scripts/evaluate.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --checkpoint {drive_checkpoint_dir}/best.pth \
    --num_classes 6 \
    --split test


## Step 7: Resume Training from Drive Checkpoint


In [ ]:
# Resume training from a checkpoint saved in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

# Check if checkpoint exists
import os
latest_checkpoint = f'{drive_checkpoint_dir}/latest.pth'
best_checkpoint = f'{drive_checkpoint_dir}/best.pth'

if os.path.exists(latest_checkpoint):
    resume_from = latest_checkpoint
    print(f'Resuming from: {resume_from}')
elif os.path.exists(best_checkpoint):
    resume_from = best_checkpoint
    print(f'Resuming from: {resume_from}')
else:
    resume_from = None
    print('No checkpoint found, starting fresh training')

# Resume training (using max_size=1024 for memory efficiency)
if resume_from:
    !python scripts/train.py \
        --dataset hituav \
        --data_dir data/hit-uav \
        --num_classes 6 \
        --batch_size 2 \
        --num_epochs 50 \
        --lr 0.005 \
        --max_size 1024 \
        --checkpoint_dir {drive_checkpoint_dir} \
        --resume {resume_from} \
        --device cuda
else:
    print('No checkpoint to resume from. Run Step 5 to start training.')


## Checkpoint Management

Checkpoints are automatically saved to Google Drive at:
`/content/drive/MyDrive/SGGF-Net-checkpoints/`

- `latest.pth` - Latest checkpoint (every epoch)
- `best.pth` - Best model based on mAP

These persist even after Colab session ends!


In [ ]:
# List checkpoints in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
import os

if os.path.exists(drive_checkpoint_dir):
    print(f"Checkpoints in Drive ({drive_checkpoint_dir}):")
    checkpoints = os.listdir(drive_checkpoint_dir)
    for ckpt in checkpoints:
        if ckpt.endswith('.pth'):
            size = os.path.getsize(f'{drive_checkpoint_dir}/{ckpt}') / (1024*1024)  # MB
            print(f"  - {ckpt} ({size:.2f} MB)")
else:
    print(f"Checkpoint directory not found: {drive_checkpoint_dir}")
    print("Run Step 5 to start training and create checkpoints.")
